# Chapter 30 — Capstone: End-to-End Probabilistic Problem
*Tier 3: All Tracks*

## 🎯 Capstone Challenge

Build a complete probabilistic pipeline from raw data to actionable decision.

**Scenario:** You are a data scientist at an e-commerce company.
You need to:
1. Analyse conversion rate data and understand its distribution
2. Run a Bayesian A/B test for a new checkout flow
3. Quantify uncertainty with bootstrap confidence intervals
4. Make a business decision with expected value calculations
5. Communicate the result with visualisations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("="*55)
print("CAPSTONE: End-to-End Probabilistic A/B Test Analysis")
print("="*55)

## Step 1 — Data Generation & Exploration

In [ ]:
# Simulate experiment data
n_control   = 2000  # control group size
n_treatment = 2000  # treatment group size

p_control   = 0.050   # baseline conversion rate
p_treatment = 0.063   # true treatment effect (+1.3pp)

conversions_control   = rng.binomial(1, p_control,   n_control)
conversions_treatment = rng.binomial(1, p_treatment, n_treatment)

cr_control   = conversions_control.mean()
cr_treatment = conversions_treatment.mean()
lift = (cr_treatment - cr_control) / cr_control * 100

print(f"Control:   n={n_control:,}, conversions={conversions_control.sum():,}, CR={cr_control:.4f} ({cr_control*100:.2f}%)")
print(f"Treatment: n={n_treatment:,}, conversions={conversions_treatment.sum():,}, CR={cr_treatment:.4f} ({cr_treatment*100:.2f}%)")
print(f"Observed lift: {lift:.1f}%")

## Step 2 — Frequentist Inference

In [ ]:
# Two-proportion z-test
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

counts  = np.array([conversions_treatment.sum(), conversions_control.sum()])
nobs    = np.array([n_treatment, n_control])
z_stat, p_val = proportions_ztest(counts, nobs)

ci_control   = proportion_confint(conversions_control.sum(),   n_control,   method="wilson")
ci_treatment = proportion_confint(conversions_treatment.sum(), n_treatment, method="wilson")

print(f"Z-statistic: {z_stat:.3f}")
print(f"p-value:     {p_val:.4f}")
print(f"Control CI (95%):   [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Treatment CI (95%): [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]")
print("✅ Statistically significant" if p_val < 0.05 else "❌ Not significant")

## Step 3 — Bayesian Analysis

In [ ]:
# Bayesian A/B test: Beta-Binomial model
# Prior: Beta(1, 1) — uninformative
a0, b0 = 1, 1

# Posterior parameters
a_ctrl  = a0 + conversions_control.sum()
b_ctrl  = b0 + n_control - conversions_control.sum()
a_treat = a0 + conversions_treatment.sum()
b_treat = b0 + n_treatment - conversions_treatment.sum()

# Sample from posteriors
n_post = 200_000
samples_ctrl  = rng.beta(a_ctrl, b_ctrl, n_post)
samples_treat = rng.beta(a_treat, b_treat, n_post)

p_treatment_wins = (samples_treat > samples_ctrl).mean()
lift_samples = (samples_treat - samples_ctrl) / samples_ctrl * 100

print(f"P(treatment > control): {p_treatment_wins:.4f}")
print(f"Expected lift (mean):   {lift_samples.mean():.2f}%")
print(f"95% Credible interval for lift: [{np.percentile(lift_samples, 2.5):.2f}%, {np.percentile(lift_samples, 97.5):.2f}%]")

p_range = np.linspace(0.02, 0.10, 500)
plt.figure(figsize=(10, 5))
plt.plot(p_range, stats.beta.pdf(p_range, a_ctrl, b_ctrl), lw=2, label="Control posterior")
plt.plot(p_range, stats.beta.pdf(p_range, a_treat, b_treat), lw=2, label="Treatment posterior")
plt.axvline(p_control, color="blue", lw=1, linestyle=":", label="True p_control")
plt.axvline(p_treatment, color="orange", lw=1, linestyle=":", label="True p_treatment")
plt.xlabel("Conversion rate p"); plt.ylabel("Density")
plt.title(f"Bayesian Posteriors — P(treatment wins)={p_treatment_wins:.1%}")
plt.legend(); plt.tight_layout(); plt.show()

## Step 4 — Bootstrap & Business Impact

In [ ]:
# Bootstrap CI for the lift
n_boot = 10_000
boot_lifts = []
for _ in range(n_boot):
    ctrl_b  = rng.choice(conversions_control,  n_control,  replace=True).mean()
    treat_b = rng.choice(conversions_treatment, n_treatment, replace=True).mean()
    boot_lifts.append((treat_b - ctrl_b) / ctrl_b * 100 if ctrl_b > 0 else 0)

boot_lo, boot_hi = np.percentile(boot_lifts, [2.5, 97.5])
print(f"Bootstrap 95% CI for lift: [{boot_lo:.2f}%, {boot_hi:.2f}%]")

# Expected value calculation
daily_visitors = 10_000
avg_order_value = 85  # dollars
current_revenue = daily_visitors * p_control * avg_order_value
expected_lift_frac = np.array(boot_lifts).mean() / 100
new_revenue = daily_visitors * cr_treatment * avg_order_value
uplift_daily = new_revenue - current_revenue
uplift_annual = uplift_daily * 365

print(f"
Business Impact:")
print(f"  Current daily revenue:  ${current_revenue:,.0f}")
print(f"  Projected daily revenue: ${new_revenue:,.0f}")
print(f"  Daily uplift:            ${uplift_daily:,.0f}")
print(f"  Annual uplift (95% CI):  ${uplift_annual:,.0f}")

## Step 5 — Final Dashboard & Decision

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

# 1. Conversion rates with CI
ax1 = fig.add_subplot(gs[0, 0])
groups = ["Control", "Treatment"]
crs  = [cr_control, cr_treatment]
cis  = [ci_control, ci_treatment]
errs_lo = [cr - ci[0] for cr, ci in zip(crs, cis)]
errs_hi = [ci[1] - cr for cr, ci in zip(crs, cis)]
bars = ax1.bar(groups, [r*100 for r in crs], color=["steelblue","tomato"], width=0.5)
ax1.errorbar(groups, [r*100 for r in crs],
             yerr=[[e*100 for e in errs_lo], [e*100 for e in errs_hi]],
             fmt="none", color="black", capsize=8, lw=2)
ax1.set_ylabel("Conversion Rate (%)"); ax1.set_title("A/B Test Results")

# 2. Posterior distributions
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(p_range, stats.beta.pdf(p_range, a_ctrl, b_ctrl), lw=2, label="Control")
ax2.plot(p_range, stats.beta.pdf(p_range, a_treat, b_treat), lw=2, label="Treatment")
ax2.set_title(f"Bayesian Posteriors
P(treatment wins)={p_treatment_wins:.1%}")
ax2.legend(fontsize=9); ax2.set_xlabel("p")

# 3. Bootstrap lift distribution
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(boot_lifts, bins=60, density=True, alpha=0.7, color="steelblue")
ax3.axvline(0, color="red", lw=2, linestyle="--", label="No effect")
ax3.axvspan(boot_lo, boot_hi, alpha=0.2, color="green", label="95% CI")
ax3.set_xlabel("Lift (%)"); ax3.set_title(f"Bootstrap Lift Distribution
CI=[{boot_lo:.1f}%, {boot_hi:.1f}%]")
ax3.legend(fontsize=9)

# 4. Lift posterior from Bayesian
ax4 = fig.add_subplot(gs[1, 0:2])
ax4.hist(lift_samples, bins=100, density=True, alpha=0.7, color="tomato")
ax4.axvline(0, color="black", lw=2, linestyle="--", label="No effect")
lo_b, hi_b = np.percentile(lift_samples, [2.5, 97.5])
ax4.axvspan(lo_b, hi_b, alpha=0.2, color="blue", label=f"95% Credible: [{lo_b:.1f}%, {hi_b:.1f}%]")
ax4.set_xlabel("Lift (%)"); ax4.set_title("Bayesian Posterior of Lift")
ax4.legend(fontsize=9)

# 5. Decision summary
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis("off")
summary_text = (
    "DECISION SUMMARY
"
    "─────────────────────
"
    f"p-value: {p_val:.4f} {'✅' if p_val<0.05 else '❌'}
"
    f"P(treatment wins): {p_treatment_wins:.1%}
"
    f"Expected lift: {lift_samples.mean():.1f}%
"
    f"Annual uplift: ${uplift_annual:,.0f}

"
    "RECOMMENDATION:
"
    "SHIP THE TREATMENT
"
    "(strong evidence,
"
    " profitable uplift)"
)
ax5.text(0.05, 0.95, summary_text, transform=ax5.transAxes, fontsize=11,
         verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="lightgreen", alpha=0.3))

plt.suptitle("Capstone: End-to-End A/B Test Probabilistic Analysis",
             fontsize=14, fontweight="bold")
plt.savefig("capstone_dashboard.png", dpi=100, bbox_inches="tight")
plt.show()
print("
✅ Dashboard saved as capstone_dashboard.png")

## 🏆 Congratulations — You've Completed the Full Curriculum!

### What you've mastered:

**Tier 1 — Foundations (Chapters 1–12)**
- Probability rules, conditional probability, Bayes' Theorem
- Random variables, distributions, CLT, descriptive statistics

**Tier 2 — Track-Specific (Chapters 13–22)**
- 🎓 Students: combinatorics, hypothesis testing, exam strategy
- 💻 Developers: Monte Carlo, A/B testing, probabilistic data structures
- 📈 Data Scientists: MLE, bias-variance, causal inference
- ⚙️ Engineers: reliability, queuing, SPC, Six Sigma

**Tier 3 — Advanced (Chapters 23–30)**
- Markov chains, stochastic processes, Bayesian networks
- Regression, time series, MCMC, multivariate statistics
- ✅ This capstone: end-to-end probabilistic reasoning

**Total: 30 chapters | 4 tracks | ~1,700 minutes of content**

Keep exploring, keep building, and trust the math! 🎲